# 03 — Matrix Factorization: SVD and SVD++

This notebook trains and compares SVD vs SVD++ models:
- Load train/val splits
- Train SVD (n_factors=100)
- Train SVD++ (n_factors=100)
- Tuning n_factors via validation RMSE
- Compare recommendation quality

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import PROCESSED_DIR, N_FACTORS, N_EPOCHS
from src.models.matrix_factorization import SVDRecommender
from src.evaluation.metrics import rmse as compute_rmse, compute_ranking_metrics
from src.evaluation.benchmark import build_ground_truth, build_user_seen_items
from src.logging_utils import model_logger

sns.set_style('whitegrid')
model_logger.start_phase('mf_nb', 'Matrix factorization notebook started')

## 1. Load Data

In [ ]:
train = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
val   = pd.read_parquet(PROCESSED_DIR / 'val.parquet')
test  = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')

## 2. Train SVD

In [ ]:
svd = SVDRecommender(n_factors=N_FACTORS, n_epochs=N_EPOCHS, use_svdpp=False)
svd.fit(train)
print(f'SVD trained: n_factors={N_FACTORS}, n_epochs={N_EPOCHS}')

# Rating prediction on val
actuals = val['rating'].tolist()
preds   = [svd.predict_rating(int(row.user_id), int(row.movie_id))
           for row in val.head(1000).itertuples()]
print(f'SVD validation RMSE (sample): {compute_rmse(actuals[:1000], preds):.4f}')

## 3. Factor Sensitivity Analysis

In [ ]:
factor_results = []
for n_fac in [20, 50, 100, 150]:
    m = SVDRecommender(n_factors=n_fac, n_epochs=10, use_svdpp=False)
    m.fit(train)
    preds = [m.predict_rating(int(r.user_id), int(r.movie_id))
             for r in val.head(500).itertuples()]
    r = compute_rmse(val['rating'].head(500).tolist(), preds)
    factor_results.append({'n_factors': n_fac, 'val_rmse': r})
    print(f'  n_factors={n_fac:3d}  val_rmse={r:.4f}')

fdf = pd.DataFrame(factor_results)
fdf.plot(x='n_factors', y='val_rmse', marker='o', title='SVD Val RMSE vs n_factors')
plt.ylabel('Val RMSE')
plt.tight_layout()
plt.show()

## 4. Train SVD++ and Compare

In [ ]:
svdpp = SVDRecommender(n_factors=N_FACTORS, n_epochs=N_EPOCHS, use_svdpp=True)
svdpp.fit(train)
print('SVD++ trained.')

ground_truth = build_ground_truth(test)
seen_items   = build_user_seen_items(train)

sample_users = list(ground_truth.keys())[:300]

svd_recs   = {uid: [r['movie_id'] for r in svd.recommend(uid, 10, seen_items.get(uid, set()))]
              for uid in sample_users}
svdpp_recs = {uid: [r['movie_id'] for r in svdpp.recommend(uid, 10, seen_items.get(uid, set()))]
              for uid in sample_users}

svd_metrics   = compute_ranking_metrics(svd_recs, ground_truth, k=10)
svdpp_metrics = compute_ranking_metrics(svdpp_recs, ground_truth, k=10)

comparison = pd.DataFrame({
    'SVD':   svd_metrics,
    'SVD++': svdpp_metrics,
}).T
print('\n--- HR@10 Comparison ---')
print(comparison[['hit_rate_at_k', 'precision_at_k', 'ndcg_at_k']].round(4))

In [ ]:
comparison[['hit_rate_at_k', 'precision_at_k', 'ndcg_at_k']].plot(
    kind='bar', title='SVD vs SVD++ (K=10)'
)
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

model_logger.end_phase('mf_nb', 'Matrix factorization notebook complete')